In [1]:
import numpy as np

In [2]:
import pandas as pd 

In [3]:
import patsy 

In [5]:
data = pd.DataFrame({
    'key1': ['a', 'a', 'b','b', 'a', 'b','a', 'b'],
    'key2': [0,1,0,1,0,1,0,0],
    'v1': [1,2,3,4,5,6,7,8],
    'v2': [-1,0,2.5,-0.5, 4.0, -1.2, 0.2, -1.7]})

In [6]:
y, X = patsy.dmatrices('v2 ~ key1', data)

In [7]:
X

DesignMatrix with shape (8, 2)
  Intercept  key1[T.b]
          1          0
          1          0
          1          1
          1          1
          1          0
          1          1
          1          0
          1          1
  Terms:
    'Intercept' (column 0)
    'key1' (column 1)

In [8]:
y, X = patsy.dmatrices('v2 ~ key1 + 0', data)

In [9]:
X

DesignMatrix with shape (8, 2)
  key1[a]  key1[b]
        1        0
        1        0
        0        1
        0        1
        1        0
        0        1
        1        0
        0        1
  Terms:
    'key1' (columns 0:2)

In [10]:
y, X = patsy.dmatrices('v2 ~ C(key2)', data)

In [11]:
X

DesignMatrix with shape (8, 2)
  Intercept  C(key2)[T.1]
          1             0
          1             1
          1             0
          1             1
          1             0
          1             1
          1             0
          1             0
  Terms:
    'Intercept' (column 0)
    'C(key2)' (column 1)

In [12]:
data['key2'] = data['key2'].map({0: 'zero', 1: 'one'})

In [13]:
data

,key1,key2,v1,v2
0,a,zero,1,-1.0
1,a,one,2,0.0
2,b,zero,3,2.5
3,b,one,4,-0.5
4,a,zero,5,4.0
5,b,one,6,-1.2
6,a,zero,7,0.2
7,b,zero,8,-1.7


In [14]:
y, X = patsy.dmatrices('v2 ~ key1 + key2', data)

In [15]:
X

DesignMatrix with shape (8, 3)
  Intercept  key1[T.b]  key2[T.zero]
          1          0             1
          1          0             0
          1          1             1
          1          1             0
          1          0             1
          1          1             0
          1          0             1
          1          1             1
  Terms:
    'Intercept' (column 0)
    'key1' (column 1)
    'key2' (column 2)

In [16]:
y, X = patsy.dmatrices('v2 ~ key1 + key2 + key1:key2', data)

In [17]:
X

DesignMatrix with shape (8, 4)
  Intercept  key1[T.b]  key2[T.zero]  key1[T.b]:key2[T.zero]
          1          0             1                       0
          1          0             0                       0
          1          1             1                       1
          1          1             0                       0
          1          0             1                       0
          1          1             0                       0
          1          0             1                       0
          1          1             1                       1
  Terms:
    'Intercept' (column 0)
    'key1' (column 1)
    'key2' (column 2)
    'key1:key2' (column 3)

In [18]:
import statsmodels.api as sm

In [19]:
import statsmodels.formula.api as smf  

In [20]:
rng = np.random.default_rng(seed=12345)

In [21]:
def dnorm(mean, variance, size = 1):
    if isinstance(size, int):
        size = size, 
        return mean + np.sqrt(variance) * rng.standard_normal(*size)

In [22]:
N = 100

In [23]:
X = np.c_[dnorm(0, 0.4, size = N),
          dnorm(0, 0.6, size = N),
          dnorm(0, 0.2, size = N)]

In [24]:
eps = dnorm(0,0.1, size = N)

In [25]:
beta = [0.1, 0.3, 0.5]


In [26]:
y = np.dot(X, beta) + eps

In [27]:
X[:5]

array([[-0.90050602, -0.18942958, -1.0278702 ],
       [ 0.79925205, -1.54598388, -0.32739708],
       [-0.55065483, -0.12025429,  0.32935899],
       [-0.16391555,  0.82403985,  0.20827485],
       [-0.04765129, -0.21314698, -0.04824364]])

In [29]:
 y[:5]

array([-0.59952668, -0.58845445,  0.18563386, -0.00747657, -0.01537445])

In [30]:
X_model = sm.add_constant(X)

In [33]:
X_model[:5]

array([[ 1.        , -0.90050602, -0.18942958, -1.0278702 ],
       [ 1.        ,  0.79925205, -1.54598388, -0.32739708],
       [ 1.        , -0.55065483, -0.12025429,  0.32935899],
       [ 1.        , -0.16391555,  0.82403985,  0.20827485],
       [ 1.        , -0.04765129, -0.21314698, -0.04824364]])

In [35]:
model = sm.OLS(y, X)

In [36]:
results = model.fit()

In [37]:
results.params

array([0.06681503, 0.26803235, 0.45052319])

In [38]:
print(results.summary())

                                 OLS Regression Results                                
Dep. Variable:                      y   R-squared (uncentered):                   0.469
Model:                            OLS   Adj. R-squared (uncentered):              0.452
Method:                 Least Squares   F-statistic:                              28.51
Date:                Thu, 13 Aug 2026   Prob (F-statistic):                    2.66e-13
Time:                        15:06:09   Log-Likelihood:                         -25.611
No. Observations:                 100   AIC:                                      57.22
Df Residuals:                      97   BIC:                                      65.04
Df Model:                           3                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

In [40]:
data = pd.DataFrame(X, columns = ['col0', 'col1', 'col2'])

In [41]:
data['y'] = y

In [42]:
data[:5]

,col0,col1,col2,y
0,-0.900506,-0.189430,-1.027870,-0.599527
1,0.799252,-1.545984,-0.327397,-0.588454
2,-0.550655,-0.120254,0.329359,0.185634
3,-0.163916,0.824040,0.208275,-0.007477
4,-0.047651,-0.213147,-0.048244,-0.015374


In [43]:
results = smf.ols('y ~ col0 + col1 + col2', data = data).fit()

In [45]:

results.params

Intercept   -0.020799
col0         0.065813
col1         0.268970
col2         0.449419
dtype: float64